# Tutorial 1: Graph Neural Network on the Cora Dataset

Author: Tri Nguyen

## Overview

In this tutorial, we will walk through the steps to build and train a Graph Neural Network (GNN) using the Cora dataset. The Cora dataset is a popular benchmark dataset for graph-based learning tasks, particularly for node classification.

### Dataset Description
The Cora dataset consists of scientific publications classified into one of seven classes. The publications are represented as a graph, where nodes correspond to papers and edges represent citations between them. Each paper is described by a bag-of-words feature vector, and the task is to predict the class label of each paper based on its features and the graph structure.

### Why Graph Neural Networks
GNNs are designed to work directly with graph-structured data and thus are a natural fit for tasks like node classification in citation networks. 
They leverage both the features of the nodes and the structure of the graph to make predictions, allowing them to capture relationships between nodes.

## Dependencies

- **Python 3.8+**
- **numpy**
- **matplotlib**
- **torch**
- **torch-geometric** (PyTorch Geometric)
  - pyg-lib
  - torch-scatter
  - torch-sparse
  - torch-cluster
  - torch-spline-conv
- **networkx** - for graph visualization
- **scikit-learn** - for metrics and t-SNE visualization

### Installing PyTorch Geometric

For GPU support, replace `cpu` with your CUDA version (e.g., `cu118` or `cu121`). Visit [PyTorch Geometric Installation](https://pytorch-geometric.readthedocs.io/en/latest/install/installation.html) for more details.

In my experience, installing PyTorch Geometric can sometimes be tricky due to compatibility issues. If you encounter problems, try installing the CPU-only version. Sometimes, `torch_geometric` and its dependencies will run into version conflicts with `torch`. I've found that installing `torch` with CUDA 12.1 (even if your CUDA version is higher) is the most reliable option, but this could depend on the system.

In [ ]:
# Install PyTorch Geometric (uncomment and run if needed)
# !pip install torch-geometric
# !pip install pyg-lib torch-scatter torch-sparse torch-cluster torch-spline-conv -f https://data.pyg.org/whl/torch-$(python -c "import torch; print(torch.__version__)")+cpu.html

In [ ]:
# Import libraries
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

## 1. Download and visualize the Cora dataset

In [ ]:
from torch_geometric.datasets import Planetoid

# Load the Cora dataset
dataset = Planetoid(root='/tmp/Cora', name='Cora')

In [ ]:
# print out dataset properties
print(f"Number of graphs: {len(dataset)}")
print(f"Number of features: {dataset.num_features}")
print(f"Number of classes: {dataset.num_classes}")

data = dataset[0]
print(f"\nGraph structure:")
print(f"  Number of nodes: {data.num_nodes}")
print(f"  Number of edges: {data.num_edges}")
print(f"  Node feature shape: {data.x.shape}")
print(f"  Edge index shape: {data.edge_index.shape}")
print(f"  Has isolated nodes: {data.has_isolated_nodes()}")
print(f"  Has self-loops: {data.has_self_loops()}")
print(f"  Is undirected: {data.is_undirected()}")

The Cora dataset includes one single large graph that contains all papers and their citation relationships.
Training, validation, and test sets are created by splitting the nodes of the graph, specifically by masking the nodes.

In [ ]:
# Examine the train/val/test split masks
print(f"Training nodes: {data.train_mask.sum().item()}")
print(f"Validation nodes: {data.val_mask.sum().item()}")
print(f"Test nodes: {data.test_mask.sum().item()}")

# Look at class distribution
class_names = ['Case_Based', 'Genetic_Algorithms', 'Neural_Networks',
               'Probabilistic_Methods', 'Reinforcement_Learning', 'Rule_Learning', 'Theory']
labels = data.y.numpy()

print("\nClass distribution:")
for i, name in enumerate(class_names):
    count = (labels == i).sum()
    print(f"  {name}: {count} papers")

We now visualize the graph using networkx, which is another popular library for graph analysis and visualization.
One of the the biggest challenges in working with graph datasets, especially large ones, is visualizing them effectively.

In [ ]:
# Visualize a subset of the graph using networkx
import networkx as nx
from torch_geometric.utils import to_networkx

# create a subgraph for visual clarity
subset_size = 500
subset_mask = torch.zeros(data.num_nodes, dtype=torch.bool)
subset_mask[:subset_size] = True
subset_edge_mask = (data.edge_index[0] < subset_size) & (data.edge_index[1] < subset_size)
subset_edge_index = data.edge_index[:, subset_edge_mask]

# create networkx graph
G = nx.Graph()
G.add_nodes_from(range(subset_size))
edges = subset_edge_index.t().numpy()
G.add_edges_from(edges)

fig, ax = plt.subplots(figsize=(8, 6))

# note that Cora nodes do not have an intrinsic position
# we will use a spring layout for visualization, which put nodes in positions based on their connectivity
pos = nx.spring_layout(G, seed=42, k=0.2)

# Color nodes by class
node_colors = data.y[:subset_size].numpy()
cmap = plt.get_cmap('plasma', dataset.num_classes)
scatter = nx.draw_networkx_nodes(
    G, pos, node_size=30, node_color=node_colors, cmap=cmap, alpha=0.8, ax=ax)
nx.draw_networkx_edges(G, pos, alpha=0.1, ax=ax)

plt.colorbar(scatter, ax=ax, label='Class')
ax.set_title(f'Cora Citation Network (first {subset_size} nodes)', fontsize=16)
ax.axis('off')
plt.tight_layout()
plt.show()

## 2. Understanding the Graph Structure

Unlike Tutorial 2 where we constructed graphs from point cloud data, the Cora dataset already comes as a graph:
- **Nodes**: Each node represents a scientific paper (2708 papers total)
- **Edges**: Edges represent citation links between papers (10556 edges)
- **Node Features**: Each paper is described by a 1433-dimensional bag-of-words feature vector
- **Labels**: Each paper belongs to one of 7 research areas

The task is *transductive node classification*: we train on a subset of labeled nodes and predict labels for the remaining nodes, all within the same graph. This differs from Tutorial 2's *inductive graph-level regression*, where we trained on multiple graphs and made predictions on new, unseen graphs.

In [ ]:
# Print the data object to see its structure
print("Data object structure:")
print(data)
print(f"\nNode features (x): {data.x.shape} - (num_nodes, num_features)")
print(f"Edge index: {data.edge_index.shape} - (2, num_edges)")
print(f"Labels (y): {data.y.shape} - (num_nodes,)")
print(f"Train mask: {data.train_mask.shape} - boolean mask for training nodes")
print(f"Val mask: {data.val_mask.shape} - boolean mask for validation nodes")
print(f"Test mask: {data.test_mask.shape} - boolean mask for test nodes")

## 3. Build and Train the GNN Model

We will use a Graph Convolutional Network (GCN) for node classification ([Kipf & Welling, 2017](https://arxiv.org/abs/1609.02907)). 
GCN is one of the foundational architectures for graph neural networks and works by aggregating features from neighboring nodes.

The message passing operation in GCN can be written as:
$$
H^{(l+1)} = \sigma\left( \mathbf{D}^{-1/2} \mathbf{A} \mathbf{D}^{-1/2} H^{(l)} W^{(l)} \right)
$$
where:
- $H^{(l)}$ is the node feature matrix at layer $l$
- $\mathbf{A}$ is the adjacency matrix (with self-loops)
- $\mathbf{D}$ is the degree matrix of $\mathbf{A}$ (with self-loops)
- $W^{(l)}$ is a learnable weight matrix
- $\sigma$ is a non-linear activation function (e.g., ReLU)

Unlike Tutorial 2 where we will use global pooling to get graph-level embeddings, here we directly use the final node embeddings for classification. Each node's output is a probability distribution over the 7 classes.

For the full list of available GNN layers, refer to: https://pytorch-geometric.readthedocs.io/en/latest/modules/nn.html

In [ ]:
from torch_geometric.nn import GCNConv
import torch.nn.functional as F

class GCN(nn.Module):
    """
    Graph Convolutional Network for node classification.
    """

    def __init__(self, input_dim, hidden_dim, output_dim, num_layers=2, dropout=0.5):
        """
        Args:
            input_dim: Dimension of input node features
            hidden_dim: Dimension of hidden layers
            output_dim: Number of output classes
            num_layers: Number of GCN layers
            dropout: Dropout probability
        """
        super().__init__()

        self.num_layers = num_layers
        self.dropout = dropout

        # GCN layers
        # feel free to modify the architecture
        # example GNN layers: GAT, GraphSAGE, GIN, GPSConv, etc.
        self.convs = nn.ModuleList()
        self.convs.append(GCNConv(input_dim, hidden_dim))
        for _ in range(num_layers - 2):
            self.convs.append(GCNConv(hidden_dim, hidden_dim))
        self.convs.append(GCNConv(hidden_dim, output_dim))

    def forward(self, data):
        """
        Forward pass through the GCN.

        Args:
            data: PyG Data object containing x and edge_index

        Returns:
            out: Node logits of shape (num_nodes, num_classes)
        """
        x, edge_index = data.x, data.edge_index

        for i, conv in enumerate(self.convs[:-1]):
            x = conv(x, edge_index)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)

        # Final layer (no activation, no dropout)
        x = self.convs[-1](x, edge_index)

        return x

# Initialize the model
model = GCN(
    input_dim=dataset.num_features,
    hidden_dim=64,
    output_dim=dataset.num_classes,
    num_layers=2,
    dropout=0.5
)
print(model)

# Verify output shape
with torch.no_grad():
    out = model(data)
    print(f"\nOutput shape: {out.shape} (num_nodes, num_classes)")

In [ ]:
def train_node_classifier(model, data, epochs=200, lr=0.01, weight_decay=5e-4, device='cpu'):
    """
    Train the GNN for node classification.

    Args:
        model: GNN model
        data: PyG Data object with train/val/test masks
        epochs: Number of training epochs
        lr: Learning rate
        weight_decay: L2 regularization strength
        device: Device to train on

    Returns:
        train_losses: List of training losses
        val_accs: List of validation accuracies
    """
    device = torch.device(device)
    model = model.to(device)
    data = data.to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.CrossEntropyLoss()

    train_losses = []
    train_accs = []
    val_losses = []
    val_accs = []

    print(f"Training for {epochs} epochs...\n")

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()

        out = model(data)
        loss = criterion(out[data.train_mask], data.y[data.train_mask])
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())

        # also compute training accuracy
        pred = out.argmax(dim=1)
        train_correct = (pred[data.train_mask] == data.y[data.train_mask]).sum()
        train_acc = train_correct / data.train_mask.sum()
        train_accs.append(train_acc.item())


        # compute the validation accuracy at the end of each epoch
        model.eval()
        with torch.no_grad():
            out = model(data)
            pred = out.argmax(dim=1)

            val_loss = criterion(out[data.val_mask], data.y[data.val_mask])
            val_losses.append(val_loss.item())

            val_correct = (pred[data.val_mask] == data.y[data.val_mask]).sum()
            val_acc = val_correct / data.val_mask.sum()
            val_accs.append(val_acc.item())

        # Print progress every 20 epochs
        if (epoch + 1) % 20 == 0:
            print(
                f"Epoch {epoch+1:3d}/{epochs} | Train Loss: {loss.item():.4f} | "\
                f"Train Acc: {train_acc:.4f} | Val Loss: {val_loss.item():.4f} | "\
                f"Val Acc: {val_acc:.4f}"
            )

    print("\nTraining complete!")
    return train_losses, train_accs, val_losses, val_accs

In [ ]:
# Train the model
train_losses, train_accs, val_losses, val_accs = train_node_classifier(
    model, data,
    epochs=200,
    lr=1e-3,
    weight_decay=5e-4,
    device='cpu'  # change to 'cuda' if GPU is available
)

In [ ]:
# plot loss and accuracy curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(train_losses, label='Train Loss')
axes[0].plot(val_losses, label='Validation Loss')
axes[0].set_xlabel('Epoch', fontsize=14)
axes[0].set_ylabel('Cross-Entropy Loss', fontsize=14)
axes[0].set_title('Training Loss', fontsize=16)

axes[1].plot(train_accs, label='Train Accuracy')
axes[1].plot(val_accs, label='Validation Accuracy')
axes[1].set_xlabel('Epoch', fontsize=14)
axes[1].set_ylabel('Accuracy', fontsize=14)
axes[1].set_title('Validation Accuracy', fontsize=16)

axes[0].legend()
axes[1].legend()

plt.tight_layout()
plt.show()

## 4. Evaluate Model Performance

Now we evaluate our trained GCN on the held-out test set. We will compute the test accuracy and visualize the confusion matrix to understand which classes the model confuses.

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Evaluate on test set
model.eval()
with torch.no_grad():
    out = model(data)
    pred = out.argmax(dim=1)

    # Test accuracy
    test_correct = (pred[data.test_mask] == data.y[data.test_mask]).sum()
    test_acc = test_correct / data.test_mask.sum()
    print(f"Test Accuracy: {test_acc:.4f}")

    # Get predictions and labels for test set
    test_pred = pred[data.test_mask].cpu().numpy()
    test_true = data.y[data.test_mask].cpu().numpy()

In [ ]:
# Print classification report
print("\nClassification Report:")
print(classification_report(test_true, test_pred, target_names=class_names))

In [ ]:
# Plot confusion matrix
cm = confusion_matrix(test_true, test_pred)

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(cm, interpolation='nearest', cmap='Blues')
ax.set_title('Confusion Matrix', fontsize=16)
plt.colorbar(im, ax=ax)

# Add labels
tick_marks = np.arange(len(class_names))
ax.set_xticks(tick_marks)
ax.set_yticks(tick_marks)
ax.set_xticklabels(class_names, rotation=45, ha='right', fontsize=10)
ax.set_yticklabels(class_names, fontsize=10)

# Add text annotations
thresh = cm.max() / 2.
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, format(cm[i, j], 'd'),
                ha='center', va='center',
                color='white' if cm[i, j] > thresh else 'black')

ax.set_ylabel('True Label', fontsize=14)
ax.set_xlabel('Predicted Label', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Visualize learned node embeddings using t-SNE
from sklearn.manifold import TSNE

# Get embeddings from the second-to-last layer
model.eval()
with torch.no_grad():
    x, edge_index = data.x, data.edge_index
    # Pass through all but the last layer to get embeddings
    for conv in model.convs[:-1]:
        x = conv(x, edge_index)
        x = F.relu(x)
    embeddings = x.cpu().numpy()

# Apply t-SNE
print("Running t-SNE on node embeddings...")
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
embeddings_2d = tsne.fit_transform(embeddings)

# Plot
fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1],
                     c=data.y.numpy(), cmap='tab10', s=10, alpha=0.7)
ax.set_xlabel('t-SNE 1', fontsize=14)
ax.set_ylabel('t-SNE 2', fontsize=14)
ax.set_title('t-SNE Visualization of Learned Node Embeddings', fontsize=16)

# Add legend
handles = [plt.scatter([], [], c=plt.cm.tab10(i/10), s=50, label=class_names[i])
           for i in range(len(class_names))]
ax.legend(handles=handles, loc='best', fontsize=10)

plt.tight_layout()
plt.show()

## Summary

In this tutorial, we built a Graph Convolutional Network (GCN) for node classification on the Cora citation network.

**Key Takeaways:**
- GNNs leverage both node features and graph structure for classification
- The t-SNE visualization shows that GCN learns meaningful representations where similar papers cluster together

**Things you should try:**
- Try changing the graph edges. For example, add self-loops or making the graph directed. You might find this PyG module useful: [https://pytorch-geometric.readthedocs.io/en/latest/modules/utils.html](https://pytorch-geometric.readthedocs.io/en/latest/modules/utils.html)
- Different GNN architectures and compare the performance. For example, [GAT](https://arxiv.org/abs/1710.10903), [GraphSAGE](https://arxiv.org/abs/1706.02216), [GIN](https://arxiv.org/abs/1810.00826) all have very different message passing operations (and thus different inductive biases).
- You can play around with other node classification datasets (e.g., CiteSeer, PubMed)